<a href="https://colab.research.google.com/github/Justee21/AdaptKit/blob/main/bedTime_LM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## bedtime-LM

Transformer trained from scratch using MoE, RoPE, KV caches, and more

## current parameters:

Tokenizer	Byte-level BPE

Embedding dimension	\(384\)

Attention heads	\(6\)

Transformer blocks	\(2\) MoE followed by
\(4\) dense

Experts per MoE block	\(8\), with \(2\) selected per token

Context length	\(128\) tokens

## Setup

imports, choosing the device, mounting drive, and setting the checkpoint path

In [ ]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, RandomSampler, Subset
from torchvision import datasets, transforms

In [ ]:
import math
import copy
import random

In [ ]:
from tokenizers import Tokenizer, decoders, pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

In [ ]:
torch.manual_seed(42)
if torch.accelerator.is_available():
  device = torch.accelerator.current_accelerator()
else:
  device = torch.device("cpu")

print(f"using {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/gpt2/weights.pth'
os.makedirs(os.path.dirname(path), exist_ok=True)

In [ ]:
path = "/content/drive/MyDrive/gpt2_moe/weights.pth"
os.makedirs(os.path.dirname(path), exist_ok=True)

In [ ]:
dtype = torch.float

## load tokenizer and data

Uses a saved tokenizer and memory-mapped training and validation files (these files are from the TinyStories Dataset uploaded from HF). Earlier data-preparation experiments are retained in the appendix.

In [ ]:
import numpy as np
tokenizer = Tokenizer.from_file(
    "/content/drive/MyDrive/tinystories/tokenizer.json"
)
vocab_size = tokenizer.get_vocab_size()

In [ ]:
training_data = np.memmap("/content/drive/MyDrive/tinystories/train.bin", dtype=np.uint16, mode="r")
validation_data = np.memmap("/content/drive/MyDrive/tinystories/valid.bin", dtype=np.uint16, mode="r")

## construct training examples

Create input sequences and next-token targets from the token files.

In [ ]:
#create a custom dataset, each user can determine the context size that they want to deal with
#Example: suppose you have a Tensor of size N and context size of length L, then the highest index that should be allowed is N-L, and then when you get an item
#it is the tensor spliced from index to context size and the target is shifted one to the right
class CharacterDataset(Dataset):
  def __init__(self, ids, context_size):
    self.ids = ids
    self.context_size = context_size
    self.total_chunks = len(self.ids) - self.context_size
  def __len__(self):
    return len(self.ids)-self.context_size
  def __getitem__(self, idx):
    #this is for the character tokenizations using text:
    # input_ids = self.ids[idx:idx+self.context_size]
    # target = self.ids[idx+1:idx+self.context_size+1]
    # return input_ids, target
    chunk = torch.tensor(
        self.ids[idx:idx + self.context_size + 1],
        dtype=torch.long,
    )
    return chunk[:-1], chunk[1:]


## define the architecture

### Causal multi-head attention and rotary positional embeddings

In [ ]:
#inputting the embedding table x with dimensions of (B,T,dmodel), we also want to have an output of same size. we have learned weight tensors WQ, WK, WV that are each dmodel x dmodel
#we then want to multiply x by each WQ, WK, WV to get Q, K, V.
#need to think about the number of heads, usually can be 8 or 16 for smaller models, more modern GPTs are more in like 32-96 range i belive from google
#idea of shaping here: X is of shape B,T,D (after the embeddings) and then we have learned matrices W_K, W_Q, W_V which in each head are of size D x d_k,
#where d_k = D/num_heads, and thus we have X@W_K is of size B,T,d_k, and then we concatenate all these and then multiply it by some W_O matrix of size B,T,D
#d_k is the head_dim here
class MultiheadAttention(nn.Module):
  def __init__(self, embed_dim, num_heads, seq_length):
    super().__init__()
    if num_heads<1:
      raise ValueError("Have to give at least 1 head")
    if embed_dim % num_heads!=0:
      raise ValueError("Embedding dimension isn't divisible by the number of heads")

    self.num_heads = num_heads
    self.head_dim = embed_dim//num_heads
    self.cached_K = None
    self.cached_V = None
    self.cache_position = 0

    self.q_layers = nn.ModuleList(
        [nn.Linear(embed_dim, self.head_dim) for _ in range(self.num_heads)]
    )
    self.v_layers = nn.ModuleList(
        [nn.Linear(embed_dim, self.head_dim) for _ in range(self.num_heads)]
    )
    self.k_layers = nn.ModuleList(
        [nn.Linear(embed_dim, self.head_dim) for _ in range(self.num_heads)]
    )
    self.out = nn.Linear(embed_dim, embed_dim)

    #since rope requires us to compute the angles where theta = 10000^-2k/d where k represents the kth 2x2 rotation block along the diagonal, we want to precompute all of the
    #k values ahead of time so we dont need to compute them in the forward pass. we need to do it hidden_dim//2 number of times
    half = self.head_dim // 2
    freqs = 1.0 / (10000 ** (torch.arange(half, dtype=torch.float32) / half))
    self.register_buffer("freqs", freqs, persistent=False)
    angles = torch.arange(seq_length, dtype=torch.float32)[:, None] * freqs[None, :]
    self.register_buffer("cos_embedding", angles.cos())
    self.register_buffer("sin_embedding", angles.sin())
    #need torch.outer to make it nxd so that we can elementwise multiply later

  def forward(self, x, use_cache=False):
    B, T, D = x.shape
    head_outputs = []
    past_length = self.cached_K[0].size(1) if use_cache and self.cached_K is not None else 0
    mask = torch.triu(torch.ones(T, past_length + T, dtype=torch.bool, device=x.device), diagonal=past_length + 1)
    new_K, new_V = [], []
    for head, (q_layer, v_layer, k_layer) in enumerate(zip(self.q_layers, self.v_layers, self.k_layers)):
      Q_i = q_layer(x)
      K_i = k_layer(x)
      V_i = v_layer(x)
      #RoPE positions keep increasing even when old cache entries are removed
      #splice into even and odd to elementwise multiply
      q_even = Q_i[:,:,::2]
      q_odd = Q_i[:,:,1::2]

      k_even = K_i[:,:,::2]
      k_odd = K_i[:,:,1::2]
      start = self.cache_position if use_cache else 0
      angles = torch.arange(start, start + T, device=x.device)[:, None] * self.freqs
      cos = angles.cos().unsqueeze(0)
      sin = angles.sin().unsqueeze(0)
      #simplify rotation matrix to elementwise multiply
      rot_q_even = q_even * cos - q_odd * sin
      rot_q_odd = q_even * sin + q_odd * cos

      rot_k_even = k_even * cos - k_odd * sin
      rot_k_odd = k_even * sin + k_odd * cos
      #we sum over the hidden dim so the order in which we keep the hidden dims in doesnt matter.
      Q_i = torch.cat((rot_q_even, rot_q_odd), dim=-1)
      K_i = torch.cat((rot_k_even, rot_k_odd), dim=-1)

      if use_cache:
        if self.cached_K is not None:
          K_i = torch.cat((self.cached_K[head], K_i), dim=1)
          V_i = torch.cat((self.cached_V[head], V_i), dim=1)
        new_K.append(K_i)
        new_V.append(V_i)

      score = Q_i @ K_i.transpose(1,2)
      scaled_score = score/math.sqrt(self.head_dim)
      masked_score = scaled_score.masked_fill(mask, -math.inf)
      softMax = torch.softmax(masked_score, dim=2)
      head_outputs.append(softMax @ V_i)
      #can replace 5 lines above with:
      #head_outputs.append(F.scaled_dot_product_attention(Q_i, K_i, V_i, attn_mask=~mask))
    if use_cache:
      self.cached_K = new_K
      self.cached_V = new_V
      self.cache_position += T
    combined = torch.cat(head_outputs, dim=2)
    return self.out(combined)




### Dense Transformer block

In [ ]:
#this is the transformer block. for each block, we pre-layer norm X to essentially normalize and then we run MHA on X. we then add X back to the original
class Transformer(nn.Module):
  def __init__(self, embed_dims, num_heads, MLP_dims, seq_length):
    super().__init__()
    self.embed_dims = embed_dims
    self.num_heads = num_heads
    self.norm_x = nn.LayerNorm(embed_dims)
    self.second_norm_x = nn.LayerNorm(embed_dims)
    self.attention = MultiheadAttention(self.embed_dims, self.num_heads, seq_length)
    self.w_up = nn.Linear(self.embed_dims, MLP_dims, bias=True)
    self.w_down = nn.Linear(MLP_dims, self.embed_dims, bias=True)

    self.w_gate_silu = nn.Sequential(
      nn.Linear(self.embed_dims, MLP_dims, bias=True),
      nn.SiLU(),
    )

    self.drop = nn.Dropout(0.2)



  def forward(self, x, use_cache=False):
    x = self.drop(self.attention(self.norm_x(x), use_cache=use_cache)) + x
    x_norm = self.second_norm_x(x)
    x_prime = self.w_up(x_norm)
    x_double_prime = self.w_down((self.w_gate_silu(x_norm) * x_prime))
    x = self.drop(x_double_prime) + x

    return x

### MoEs Transformer block

In [ ]:
class MoETransformer(nn.Module):
  def __init__(self, embed_dims, num_heads, MLP_dims, seq_length, top_k, num_experts):
    super().__init__()
    MLP_dims//=top_k
    self.embed_dims = embed_dims
    self.num_heads = num_heads
    self.norm_x = nn.LayerNorm(embed_dims)
    self.second_norm_x = nn.LayerNorm(embed_dims)
    self.attention = MultiheadAttention(self.embed_dims, self.num_heads, seq_length)
    self.w_ups = nn.ModuleList([nn.Linear(self.embed_dims, MLP_dims, bias=True) for _ in range(num_experts)])
    self.w_downs = nn.ModuleList([nn.Linear(MLP_dims, self.embed_dims, bias=True) for _ in range(num_experts)])

    self.router = nn.Linear(embed_dims, num_experts)
    self.swiglus = nn.ModuleList([nn.Sequential(
      nn.Linear(self.embed_dims, MLP_dims, bias=True),
      nn.SiLU(),
    ) for _ in range(num_experts)])

    # self.w_gate_silu = nn.Sequential(
    #   nn.Linear(self.embed_dims, MLP_dims, bias=True),
    #   nn.SiLU(),
    # )
    self.top_k = top_k
    self.drop = nn.Dropout(0.2)
    self.num_experts = num_experts



  def forward(self, x, use_cache=False):
    x = self.drop(self.attention(self.norm_x(x), use_cache=use_cache)) + x
    x_norm = self.second_norm_x(x)

    x_moe = self.router(x_norm)
    values, ids = torch.topk(x_moe, k=self.top_k, dim=-1)

    router_probs = torch.softmax(x_moe.float(), dim=-1)
    P = router_probs.mean(dim=(0, 1))
    counts = torch.bincount(
        ids.reshape(-1), minlength=self.num_experts
    )
    f = counts.float() / ids.numel()
    aux_loss = self.num_experts * (f * P).sum()

    weights = torch.softmax(values, dim=-1)
    x_after_moe = torch.zeros_like(x_norm)
    for expert in range(self.num_experts):
      b, t, slot = torch.where(ids==expert)
      tokens = x_norm[b, t]
      x_prime = self.w_ups[expert](tokens)
      x_double_prime = self.w_downs[expert]((self.swiglus[expert](tokens) * x_prime))
      weighted = weights[b, t, slot].unsqueeze(-1) * x_double_prime
      x_after_moe.index_put_((b, t), weighted, accumulate=True)
    x = self.drop(x_after_moe) + x

    return x, aux_loss

## complete model

In [ ]:
#here, we defined the embedding table and positional encoding table. in the beginning, they are completely random since these weights are learned and what we will be optimizing.
#the input_ids represent the batch of sequences that we want to train over where B represents the batch size and T represents the length of the sequence.
#for each token in our input, we then fetch the corresponding embedding tensor from the embedding table.
#for positional encodings, these just correspond to positions in the training examples, meaning each token at index i in each sequence should have the same encoding across batches
#the biggest challenge we had here was understanding the embedding table and understanding how to use it properly
class Model(nn.Module):
  def __init__(self, vocab_size, embed_dims, seq_length, num_heads, moe_layers, normal_layers, MLP_dims, top_k, num_experts):
    super().__init__()
    self.vocab_size = vocab_size
    self.seq_length = seq_length
    self.embedding = nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=embed_dims)
    self.trans_blocks = nn.ModuleList(
        [MoETransformer(embed_dims, num_heads, MLP_dims, seq_length, top_k, num_experts) for _ in range(moe_layers)]+[Transformer(embed_dims, num_heads, MLP_dims, seq_length) for _ in range(normal_layers)]
    )
    self.embed_drop = nn.Dropout(0.2)
    self.final_layer_norm = nn.LayerNorm(embed_dims)
    self.final_linear = nn.Linear(embed_dims, self.vocab_size, bias=False)
    #maps the (B,T,V) to an embedding table (V,d_model)
    self.final_linear.weight = self.embedding.weight
    #so we have the same matrix and weights that bring the token ids to the vector embedding as the weights to bring the token ids to the vector embedding
    nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)



  def forward(self, input_ids, use_cache=False):
    B, T = input_ids.shape
    embed = self.embedding(input_ids)
    x = self.embed_drop(embed)
    aux_losses = []
    for block in self.trans_blocks:
        if isinstance(block, MoETransformer):
            x, block_aux = block(x, use_cache=use_cache)
            aux_losses.append(block_aux)
        else:
            x = block(x, use_cache=use_cache)
    aux_loss = (
        torch.stack(aux_losses).mean()
        if aux_losses else x.new_zeros(())
        )
    x = self.final_layer_norm(x)
    logits = self.final_linear(x)
    return logits, aux_loss


## define the training and validation

(defines validation_loop before training_loop since the validation is tested throughout the training loop)

In [ ]:
def validation_loop(dataloader, model, loss_func, batch_size):
  model.eval()
  totalLoss = 0
  top_one_correct = 0
  top_three_correct = 0
  size = 0
  for batch, (input_ids, target) in enumerate(dataloader):
    input_ids = input_ids.to(device)
    target = target.to(device)
    with torch.no_grad():
      size+=target.numel() #counts the number of elements in target, which is B x T, which gives the number of tokens
      logits, aux_loss = model(input_ids)
      totalLoss += loss_func(logits.transpose(1,2), target).item() * target.numel()
      selected_token = logits.argmax(dim=-1)
      top_three_selected_tokens = torch.topk(logits, 3, dim=-1).indices
      top_three_correct += (top_three_selected_tokens == target.unsqueeze(-1)).any(dim=-1).sum().item()
      #basically because we have some B,T,3 vector that we want to see if each b,t vector has one of the values the same as the target so we have to put the target into the B,T,1 form
      #to compare the vectors and then we check if the target is equal to any one of those 3 items in the top_three_selected_tokens and if so we say yes and that's true and then we add all
      top_one_correct+= (selected_token==target).sum().item()
  print(f"total loss: {totalLoss/size}, accuracy of next token prediction: {top_one_correct/size}, \n accuracy that one of the top 3 predicted tokens is correct: {top_three_correct/size}")
  return totalLoss/size

In [ ]:
#training
#need to think about which sequences I want to train on since could be computationally heavy and have a lot of overlap if i do for every context_size sequence, right now
#not worrying about it but can in the future maybe make random batched starting indices - this should be done while processnig the data
def train_loop(dataloader, model, optimizer, loss_func, batch_size, epochs, early_stopping_size):
  model.train()
  min_val_loss = math.inf
  curr = 0
  steps = 0
  best_parameters = copy.deepcopy(model.state_dict())
  for i in range(epochs):
    for batch, (input_ids, target) in enumerate(dataloader):
      input_ids = input_ids.to(device)
      target = target.to(device)
      optimizer.zero_grad()
      logits, aux_loss = model(input_ids)
      ce_loss = loss_func(logits.transpose(1, 2), target)
      loss = ce_loss + 0.01 * aux_loss
      loss.backward()
      optimizer.step()
      steps+=1
      if batch%100==0:
        print(f"{i}, {batch}, {loss}")
      if steps%250==0:
        val_loss = validation_loop(validation_loader, model, loss_func, batch_size) #to prevent overfitting to the training dataset
        model.train()
        if val_loss<min_val_loss-early_stopping_size:
          curr=0
        else:
          curr+=1
        if val_loss<min_val_loss:
          min_val_loss = val_loss
          best_parameters = copy.deepcopy(model.state_dict())
        if curr == 5:
          model.load_state_dict(best_parameters)
          return min_val_loss
  model.load_state_dict(best_parameters)
  return min_val_loss


#with our first model, after running with batch size 64 and 10 epochs, we got 7800, 2.358741283416748
#took 40+ minutes running on v5e-1 TPU (no cuda)

## configuring the experiment

constructs the data loaders, model, loss function, and optimizer. Running this section initializes a new model and optimizer.

In [ ]:
batch_size = 64
max_steps = 25000
context_size = 128

new_training_data = CharacterDataset(training_data, context_size)
new_validation_data = CharacterDataset(validation_data,context_size)

sampler = RandomSampler(new_training_data, replacement=True, num_samples = max_steps*batch_size)

training_loader = DataLoader(new_training_data, batch_size=batch_size, sampler=sampler)
#this is to act more similarly to nanogpt and choose randomly
#selected indices with replacement, idea here is choose batch_size * max_steps samples which here is 64*5000=320,000
#so we have that many sequences we check through


generator = torch.Generator().manual_seed(123) #this allows for more optimal
val_indices = torch.randint(high=len(new_validation_data), size=(20* batch_size,), device="cpu", generator=generator).tolist()
#this gives random but reproducible validation indices to test on

validation_loader = DataLoader(Subset(new_validation_data, val_indices), batch_size=batch_size, shuffle=False)
#this can just be used during training_loop and separate full validation test can be evaluated later

model = Model(
    vocab_size=vocab_size, embed_dims=384,
    seq_length=128, MLP_dims=4*384, num_heads=6,
    moe_layers=2, normal_layers=4,
    top_k=2, num_experts=8,
).to(device) #hyperparameters
loss_func = nn.CrossEntropyLoss() #design  choice here
optimizer = torch.optim.AdamW(model.parameters(), lr=.0001) #design choice here

## train and save

This section trains a new model and saves its checkpoint. To use an existing checkpoint, skip this training cell and continue to the checkpoint-loading section.

In [ ]:
train_loop(
    training_loader, model, optimizer, loss_func,
    batch_size, 1, .001
)

torch.save({
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
}, path)

## load a checkpoint

Use this section to load the checkpoint at the configured path for evaluation or generation. If you are continuing directly from training, you can proceed to evaluation.

In [ ]:
#to load the weights for the model
#only need this for inference
weights = torch.load(path, map_location=device)
model.load_state_dict(weights["model"])
model.to(device)

## evaluation

Evaluate the current model on the validation subset. Saved outputs are retained from earlier executions.

In [ ]:
validation_loop(validation_loader, model, loss_func, 64)
#for batchsize = 64, embed_dims = 48, seq_length=128, num_heads=4, num_layers=3
#total loss: 2.3619683620955656, accuracy of next token prediction: 0.3068228035355258,
#accuracy that one of the top 3 predicted tokens is correct: 0.5473197877024019

## inference!

define the generation, then run the example prompt using the current model and tokenizer.

In [ ]:
def inference_loop(prompt, model, temperature, seq_length, new_tokens):
  model.eval()
  if temperature <= 0:
    print("temperature must not be 0 or negative")
    return

  prompt_ids = tokenizer.encode(prompt).ids


  prompt_ids = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)

  for i in range(new_tokens):
    prompt_ids = prompt_ids.to(device)
    with torch.no_grad():
      new_prompt_ids = prompt_ids[:, -seq_length:]
      if i == 0:
        #start fresh only for a new prompt
        for block in model.trans_blocks:
          block.attention.cached_K = None
          block.attention.cached_V = None
          block.attention.cache_position = 0
      else:
        new_prompt_ids = prompt_ids[:, -1:] #only process the new token
        if prompt_ids.size(1) > seq_length:
          #remove the oldest entry to make room for the new token
          for block in model.trans_blocks:
            block.attention.cached_K = [
                k[:, 1:].clone() for k in block.attention.cached_K
            ]
            block.attention.cached_V = [
                v[:, 1:].clone() for v in block.attention.cached_V
            ]
      logits, aux_loss = model(new_prompt_ids, use_cache=True)
      new_logits = logits[:, -1, :] #gives us last logits for the tokens
      new_logits = new_logits / temperature
      probs = torch.softmax(new_logits, dim=-1)
      new_token = torch.multinomial(probs, num_samples=1)
      prompt_ids = torch.cat((prompt_ids, new_token), dim=-1)

  #need to convert from ids to actual characters. i made a new list idx_to_char to access the chars associated to the index
  ids = torch.squeeze(prompt_ids, dim=0).tolist()
  print(tokenizer.decode(ids))

#made the model generate 100 new tokens and we got this:
#Shall I compare thee to a summer's day? Thou art more lovely and more temperate: Rough winds do shake the darling buds of May,
#yomeroWh cot Of Inovey lld whoust isilche,
#Gice th! O artensou, pr lthouscaf aced Cawik an trye,
#Wh

In [ ]:
prompt = "Tom and Jane are friends. One day, Jane goes to Tom’s house. Tom has a big pot of soup. He wants to share it with Jane. “Jane, do you want some soup?” Tom asks. Yes, please. It looks yummy, Jane says. Tom pours some soup into two bowls. He gives one bowl to Jane. Jane takes a spoonful of soup, but then she makes a face. The soup is"
inference_loop(prompt, model, .7, 128, 500)

## appendix: other experiments and checkpoints

These cells below are separate from the main workflow and are just meant to be references to what's been done earlier. Run only the cells needed for a specific experiment becuase executing the entire appendix can overwrite the active tokenizer, data variables, or model state.

## earlier data prep

This route writes local tokenizer and token files, while the main workflow reads files from Drive. Review file locations and tokenizer compatibility before using it.

In [ ]:
#importing the data (this is tinyshakespeare)
import requests
# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
#tiny stories
url = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt"
text = requests.get(url).text

In [ ]:
vocab_size = 2048

In [ ]:
#general purpose tokenizer
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = BpeTrainer(vocab_size = vocab_size, min_frequency=2, special_tokens=["<bos>", "<eos>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),)
tokenizer.train_from_iterator([text], trainer=trainer)
tokenizer.save('tokenizer.json')

ids = tokenizer.encode('hello world!').ids
print(ids, tokenizer.decode(ids))
data = torch.tensor(tokenizer.encode(text[:int(.9*len(text))]).ids, dtype = torch.long)
n = len(data)

In [ ]:
from itertools import islice
from huggingface_hub import hf_hub_download
import numpy as np

repo = "roneneldan/TinyStories"
train_path = hf_hub_download(
    repo_id=repo, repo_type="dataset",
    filename="TinyStoriesV2-GPT4-train.txt",
)
valid_path = hf_hub_download(
    repo_id=repo, repo_type="dataset",
    filename="TinyStoriesV2-GPT4-valid.txt",
)
print("starting tokenizer", flush=True)
with open(train_path, encoding="utf-8") as file:
    tokenizer.train_from_iterator(islice(file, 200_000), trainer=trainer)
tokenizer.save("tokenizer.json")
print("encoding train file", flush=True)
def make_token_file(input_path, output_path):
    buffer = []
    with open(input_path, encoding="utf-8") as source, open(output_path, "wb") as output:
        for line in source:
            buffer.extend(tokenizer.encode(line).ids)
            if len(buffer) >= 1_000_000:
                np.asarray(buffer, dtype=np.uint16).tofile(output)
                buffer.clear()
        if buffer:
            np.asarray(buffer, dtype=np.uint16).tofile(output)

make_token_file(train_path, "train.bin")
make_token_file(valid_path, "valid.bin")

### earlier tokenization and batching experiments

In [ ]:
# #how to improve this: usual a real dataset maybe
# #processing the data (matching each character with an id via ord('char')-ord('a'))
# characters = sorted(list(set(text)))
# charToIdx = {}
# for idx, char in enumerate(characters):
#   charToIdx[char]=idx

# idx_to_char = [None] * len(charToIdx)
# for char, idx in charToIdx.items():
#   idx_to_char[idx] = char

# #so now I want to create a tensor that is of size length of the text input and the ith element in the tensor represents the ID of the ith element in the text
# #just wanted to start off with each character representing a token, but can move to a more complex BPE tokenization
# IDlist = []
# for char in text:
#   IDlist.append(charToIdx[char])
# data = torch.tensor(IDlist)

# n = len(IDlist)
# training_data = data[:int(.9*n)]
# validation_data = data[int(.9*n):]

In [ ]:
training_data = data[:int(.9*n)]
validation_data = data[int(.9*n):]

In [ ]:
#how to improve this: start with a custom or real dataset/dataloader class
inputs = []
targets = []
for i in range(5):
  i*=1000
  inputs.append(training_data[i:128+i])
  targets.append(training_data[i+1:128+i+1])
inputs = torch.stack(inputs)
targets = torch.stack(targets)
# def decode(idx):
#   return characters[idx]
# print("".join(map(decode, inputs[0].tolist())))
# print("\n \n \n")
# print("".join(map(decode, targets[0].tolist())))


### locate existing token files

In [ ]:
!find /content/drive/MyDrive -name 'train.bin' -o -name 'valid.bin'

### additional save and load examples

These duplicate examples are retained from development. The older loader below uses the gpt2 checkpoint directory rather than the main gpt2_moe path.

In [ ]:
torch.save({
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
}, path)

In [ ]:
#to save the weights so dont have to run it every time to have good weighs
#ONLY RUN THIS AFTER RETRAINING
torch.save({"model": model.state_dict(),
            "optimizer": optimizer.state_dict()}, path)

In [ ]:
weights = torch.load(path, map_location=device)
model.load_state_dict(weights["model"])

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(
    "/content/drive/MyDrive/tinystories/tokenizer.json"
)

weights = torch.load(
    "/content/drive/MyDrive/gpt2/weights.pth",
    map_location=device
)
model.load_state_dict(weights["model"])
model.eval()

In [ ]:
#current improvements from v1: using cuda, changing the computationally heavy rerunning on every
#sequence so now instead of looking at every sequence we look at sequences that differ by the context size so no more overlap (also possible to randomly sample this, we should check out which is better)